In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from IPython.display import Markdown, display
from dotenv.ipython import load_dotenv

In [2]:
load_dotenv(override=True)


True

In [3]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [4]:
agent = create_agent(
    model=llm, 
    system_prompt= "You are a helpful assistant",
)

In [5]:
resp = agent.invoke(input={
    "messages": [
        {"role": "user", "content": "Je m'appelle Ouss"}
    ]
})

In [6]:
print(resp["messages"][-1].content)

Bonjour Ouss ! Comment puis-je vous aider aujourd'hui ?


In [7]:
resp = agent.invoke(input={
    "messages": [
        {"role": "user", "content": "comment je m'appelles?"}
    ]
})

In [8]:
print(resp["messages"][-1].content)

Je suis désolé, mais je ne connais pas ton nom. Si tu veux bien me le dire, je serai ravi de m'adresser à toi par ton prénom.


In [9]:
from langchain.agents.middleware import ModelRequest, ModelResponse,wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [10]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [11]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.runtime.context.get("env", "test")
    
    if env == "prod":
        model = advanced_llm
        print("Using advanced model for production")
    else:       
        model = basic_llm
        print("Using basic model for testing")
   
    return handler(request.override(model=model))

In [12]:
dynamic_agent = create_agent(
    model=basic_llm,
    tools=[], 
    middleware=[dynamic_model_selection],
    debug=True,
)

In [13]:
response = dynamic_agent.invoke(
    input={"messages": [ HumanMessage("Quel est le modèle utilisé?")]},
    context={"env": "prod"} 
)

[values] {'messages': [HumanMessage(content='Quel est le modèle utilisé?', additional_kwargs={}, response_metadata={}, id='513b98d4-c7b6-4497-8a83-332eafda6b64')]}
Using advanced model for production
[updates] {'model': {'messages': [AIMessage(content="Je suis basé sur le modèle GPT-3 de OpenAI. Ce modèle utilise l'architecture de réseau de neurones appelée Transformer pour traiter et générer du texte de manière cohérente et contextuelle. Si vous avez des questions ou avez besoin d'informations supplémentaires, n'hésitez pas à demander!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 13, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9894c391cd', 'id': 'ch

In [14]:
print(display(Markdown(response['messages'][-1].content)))

Je suis basé sur le modèle GPT-3 de OpenAI. Ce modèle utilise l'architecture de réseau de neurones appelée Transformer pour traiter et générer du texte de manière cohérente et contextuelle. Si vous avez des questions ou avez besoin d'informations supplémentaires, n'hésitez pas à demander!

None


In [15]:
agent_memory = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpful assistant",
)

In [16]:
print(resp["messages"][-1].content)

Je suis désolé, mais je ne connais pas ton nom. Si tu veux bien me le dire, je serai ravi de m'adresser à toi par ton prénom.


In [17]:
resp = agent_memory.invoke(input={"messages": [HumanMessage("Comment je m'appelle?")]})

In [18]:
print(resp["messages"][-1].content)

Je suis désolé, mais je ne sais pas comment vous vous appelez. Si vous partagez votre nom avec moi, je pourrai vous aider en utilisant ce nom dans nos échanges.


In [19]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent_memory = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpful assistant",
    checkpointer=memory,
)

In [20]:
config = {"configurable": { "thread_id": 1 }} 
resp = agent_memory.invoke(
    input={"messages": [HumanMessage("Je m'appelle Ouss")]},
    config=config
)

In [21]:
print(resp["messages"][-1].content)

Bonjour Ouss ! Comment puis-je vous aider aujourd'hui ?


In [22]:
resp = agent_memory.invoke(
    input={"messages": [HumanMessage("Je m'appelle Ouss")]},
    config=config
)

In [23]:
print(resp["messages"][-1].content)

Bonjour Ouss ! Que puis-je faire pour vous aujourd'hui ?


In [24]:
from langchain.tools import tool

In [25]:
@tool("get_meteo")
def get_meteo(city: str) -> str:
    """
    Get the current weather of the given city.
    """
    print("Whether tool called with city:", city)
    return {
        "city": city,
        "temperature": "25°C",
        "condition": "Sunny",
        "pression": "1013 hPa",
    }

In [26]:
@tool("get_employee_info")
def get_employe_info(employee_name: str) -> str:
    """
    Get employee infos about the given employee
    """
    print("Employee info tool called with name:", employee_name)
    return {
        "name": employee_name,
        "position": "Software Engineer",
        "department": "Engineering",
        "salary": 100000,
        "seniority": 5,
    }

In [27]:

config = {"configurable": { "thread_id": 1 }}
agent_with_tools = create_agent(
    model=basic_llm,
    tools=[get_meteo, get_employe_info],
    checkpointer=memory,
    system_prompt="answer user questions using the tools provided tools",
)

In [28]:
resp = agent_with_tools.invoke(
    input={"messages": [HumanMessage("la meteo à casablanca?")]},
    config=config
 )
print(display(Markdown(resp["messages"][-1].content)))

Whether tool called with city: Casablanca


La météo à Casablanca est actuellement ensoleillée avec une température de 25°C et une pression de 1013 hPa. Avez-vous besoin d'autres informations ?

None


In [29]:
resp = agent_with_tools.invoke(
    input={"messages": [HumanMessage("le salaire de Nizar?")]},
    config=config
 )
print(display(Markdown(resp["messages"][-1].content)))

Employee info tool called with name: Nizar


Nizar est ingénieur logiciel dans le département d'ingénierie. Son salaire est de 100 000. Avez-vous besoin d'autres informations ?

None


In [30]:
from ddgs import DDGS
@tool
def web_search(query: str, num_results:int=5) -> str:
    """
    Search the web usin DuckDuckGo
    Args:
    query : Search query string
    num_results : Number of results to return (Default=5)
    Returns:
    Formatted search results with titles, descptions and Urls
    """
    try:
        print(f'Search tool is called for {query}')
        ddgs_search = DDGS()
        results=ddgs_search.text(query=query, max_results=num_results, backend="google")
        if not results:
            return f"No results found for {query} "
        formatted_results = [f"Search for {query} : \n"]
        for i, result in enumerate(results,1):
            title = result.get("title","No Title")
            body = result.get("body","No description available")
            href = result.get("href","")
            formatted_results.append( f"{i}. **{title}**. {body}. {href}")
        return formatted_results
    except Exception as e:
        print(str(e))

In [31]:
agent = create_agent(model=basic_llm, tools=[web_search, get_employe_info,get_meteo], debug=True)
resp=agent.invoke(input={'messages':[HumanMessage('Search for python tutorials')]})
print(display(Markdown(resp['messages'][-1].content)))

[values] {'messages': [HumanMessage(content='Search for python tutorials', additional_kwargs={}, response_metadata={}, id='f49d54a5-0b8f-462c-8f79-6f410960d4b1')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 149, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5086b7b9a', 'id': 'chatcmpl-DRJkIpd6UECYHkcniwxvF4F4ohENS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5e45-e0fa-7c90-9164-31cc1398d249-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'python tutorials'}, 'id': 'call_r5jHsYRu04dh1wntJdAjlvJP', 'type': 'tool_call'}], invalid_tool_

Here are some Python tutorials you can explore:

1. **[Python Tutorial - W3Schools](https://www.w3schools.com/python/)**  
   Learn Python, a popular programming language that can be used to create web applications.

2. **[The Python Tutorial — Python 3.14.3 documentation](https://docs.python.org/3/tutorial/index.html)**  
   An informal introduction to the basic concepts and features of the Python language and system.

3. **[Real Python: Python Tutorials](https://realpython.com/)**  
   Offers Python tutorials for developers of all skill levels, along with books, courses, news, code examples, and articles.

4. **[Python Tutorial](https://www.pythontutorial.net/)**  
   A comprehensive tutorial that helps you understand the Python programming language and its concepts deeply.

5. **[Learn Python - Free Interactive Python Tutorial](https://www.learnpython.org/)**  
   Get started with Python through interactive coding challenges and videos, provided by DataCamp.

Feel free to check them out!

None


In [32]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
import asyncio
from datetime import datetime

In [33]:
tavily = TavilySearch(max_results=10, search_depth="advanced")


In [ ]:
@tool
def search_web(query: str):
    """
    Search for general  current web results
    """
    print("Search Tool invoked")
    results = tavily.invoke({"query": query})
    return results

In [35]:
model = init_chat_model("gpt-4o", model_provider="openai", temperature=0)
agent = create_agent(
    model=model, 
    tools=[search_web], 
    system_prompt=f"""You are a helpful assistant that serach the web
    for information using search tool
    today's date is {datetime.now().strftime("%Y-%m-%d")}
    """,
    debug=True,
)

In [36]:
resp = agent.invoke(
    input={'messages':[HumanMessage("Quel temps fait-il aujourd'hui à Casablanca")]},
    config=config)


[values] {'messages': [HumanMessage(content="Quel temps fait-il aujourd'hui à Casablanca", additional_kwargs={}, response_metadata={}, id='ba1efbf0-cc06-471c-a65f-7d549a4ae64c')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 83, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c8b70290c4', 'id': 'chatcmpl-DRJkpvQz4Vl9do97UjRArkUCphUbe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5e46-6248-7172-a1b1-c7ace274ca6e-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'météo Casablanca 5 avril 2026'}, 'id': 'call_hB1qdNFqqK02YyDtWillGdcN', 'type': 'tool

In [37]:
print(display(Markdown(resp['messages'][-1].content)))

Aujourd'hui, le 5 avril 2026, à Casablanca, le temps est ensoleillé avec des températures maximales autour de 26°C et minimales autour de 16°C. Le vent souffle à environ 13 km/h. L'humidité est d'environ 49%. Le ciel est principalement dégagé, offrant une journée agréable et ensoleillée.

None


In [38]:
from langchain_experimental.tools import PythonREPLTool

In [39]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [40]:
repl_agent = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""Génère et exécuter le code python en placant le code python et le résultat dans le fichier doc.txt """,
    debug=True,
)

In [41]:
response = repl_agent.invoke(
    input={"messages": [ HumanMessage("Je veux trier les deux listes suivantes [3, 1, 2] et [45,788,12] et puis calculer la somme des deux listes triees  ")]},
)

[values] {'messages': [HumanMessage(content='Je veux trier les deux listes suivantes [3, 1, 2] et [45,788,12] et puis calculer la somme des deux listes triees  ', additional_kwargs={}, response_metadata={}, id='88bcd68b-d70d-4c5c-a045-bcf92348134a')]}


Python REPL can execute arbitrary code. Use with caution.


[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 140, 'total_tokens': 294, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f84913767', 'id': 'chatcmpl-DRJlCT7yIZX8aRAaNnA4oDno8pgNC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5e46-bc40-7610-8706-863e99e5035c-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'list1 = [3, 1, 2]\nlist2 = [45, 788, 12]\nsorted_list1 = sorted(list1)\nsorted_list2 = sorted(list2)\nprint(sorted_list1)\nprint(sorted_list2)'}, 'id': 'call_x2xEk4Uj6TRtmuZPXb63eoYt', 'type': 'tool_call'}, {'name': 'Python_REPL', 'args': {'query': 'sorted_list1

In [42]:
print(display(Markdown(response['messages'][-1].content)))

Le code et le résultat ont été enregistrés dans le fichier `doc.txt`.

None
